##Model 1: Transformer (DEIT)
This code file implements a binary image classification pipeline using a transformer model (DeiT) to classify images as silver or non-silver. It loads images from Google Drive, balances the dataset by sampling equal classes, and splits the data into training, validation, and test sets. A custom dataset class handles image loading, preprocessing, and light augmentation. The model is fine-tuned using Hugging Face’s Trainer with early stopping and evaluated using accuracy, precision, recall, and F1 score. The model is tested, and performance is analyzed using a confusion matrix and classification report, with additional threshold tuning applied to optimize F1 score.

In [2]:
# =========================
# Install
# =========================
!pip install -q transformers accelerate evaluate pillow scikit-learn

# =========================
# Imports
# =========================
import os
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from transformers import (
    AutoImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# =========================
# Google Drive
# =========================
drive.mount("/content/drive")

# =========================
# Paths
# =========================
silver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/silver_images"
nonsilver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/non_silver_images"

SEED = 42
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================
# Checkpoint
# IMPORTANT:
# For this checkpoint, use ViTForImageClassification.
# =========================
CHECKPOINT = "facebook/deit-small-patch16-224"

# =========================
# Processor + model
# =========================
image_processor = AutoImageProcessor.from_pretrained(CHECKPOINT)

model = ViTForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label={0: "non_silver", 1: "silver"},
    label2id={"non_silver": 0, "silver": 1},
    ignore_mismatched_sizes=True,  # only classifier head should be resized
)

# =========================
# File lists
# =========================
silver_files = [
    f for f in os.listdir(silver_dir)
    if os.path.isfile(os.path.join(silver_dir, f))
]

nonsilver_files = [
    f for f in os.listdir(nonsilver_dir)
    if os.path.isfile(os.path.join(nonsilver_dir, f))
]

print("Silver files:", len(silver_files))
print("Non-silver files:", len(nonsilver_files))

# =========================
# Sample equal non-silver
# =========================
sampled_nonsilver_files = random.sample(nonsilver_files, len(silver_files))

silver_paths = [os.path.join(silver_dir, f) for f in silver_files]
nonsilver_paths = [os.path.join(nonsilver_dir, f) for f in sampled_nonsilver_files]

random.shuffle(silver_paths)
random.shuffle(nonsilver_paths)

# =========================
# Split each class separately
# =========================
n_per_class = len(silver_paths)

test_count = int(n_per_class * TEST_SPLIT)
val_count = int(n_per_class * VAL_SPLIT)

silver_test = silver_paths[:test_count]
silver_val = silver_paths[test_count:test_count + val_count]
silver_train = silver_paths[test_count + val_count:]

nonsilver_test = nonsilver_paths[:test_count]
nonsilver_val = nonsilver_paths[test_count:test_count + val_count]
nonsilver_train = nonsilver_paths[test_count + val_count:]

# =========================
# Combine splits
# =========================
train_paths = silver_train + nonsilver_train
train_labels = [1] * len(silver_train) + [0] * len(nonsilver_train)

val_paths = silver_val + nonsilver_val
val_labels = [1] * len(silver_val) + [0] * len(nonsilver_val)

test_paths = silver_test + nonsilver_test
test_labels = [1] * len(silver_test) + [0] * len(nonsilver_test)

# =========================
# Shuffle within each split
# =========================
train_combined = list(zip(train_paths, train_labels))
val_combined = list(zip(val_paths, val_labels))
test_combined = list(zip(test_paths, test_labels))

random.shuffle(train_combined)
random.shuffle(val_combined)
random.shuffle(test_combined)

train_paths, train_labels = zip(*train_combined)
val_paths, val_labels = zip(*val_combined)
test_paths, test_labels = zip(*test_combined)

train_paths, train_labels = list(train_paths), list(train_labels)
val_paths, val_labels = list(val_paths), list(val_labels)
test_paths, test_labels = list(test_paths), list(test_labels)

print("Train size:", len(train_paths))
print("Val size:", len(val_paths))
print("Test size:", len(test_paths))

# =========================
# Dataset
# =========================
class SilverDataset(Dataset):
    def __init__(self, paths, labels, processor, train=False):
        self.paths = paths
        self.labels = labels
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]

        image = Image.open(path).convert("RGB")

        # light augmentation for training
        if self.train:
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)

        encoded = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

train_ds = SilverDataset(train_paths, train_labels, image_processor, train=True)
val_ds = SilverDataset(val_paths, val_labels, image_processor, train=False)
test_ds = SilverDataset(test_paths, test_labels, image_processor, train=False)

# =========================
# Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    p = precision_score(labels, preds, zero_division=0)
    r = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
    }

# =========================
# Training arguments
# =========================
training_args = TrainingArguments(
    output_dir="./deit_silver_output",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

# =========================
# Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================
# Train
# =========================
trainer.train()

# =========================
# Validation results
# =========================
val_results = trainer.evaluate()
print("\nValidation Results:")
for k, v in val_results.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# =========================
# Test prediction
# =========================
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

# =========================
# Classification report at threshold 0.5
# =========================
y_pred = (probs > 0.5).astype(int)

print("\nConfusion Matrix (threshold = 0.50):")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report (threshold = 0.50):")
print(classification_report(y_true, y_pred, digits=4))

# =========================
# Threshold tuning
# =========================
best_t, best_f1 = 0.5, 0.0

for t in np.arange(0.20, 0.70, 0.01):
    preds = (probs > t).astype(int)

    f1 = f1_score(y_true, preds)
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)

    print(f"t={t:.2f} | F1={f1:.2f} | P={p:.2f} | R={r:.2f}")

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"\nBest threshold: {best_t:.2f} | Best F1: {best_f1:.2f}")

y_pred_best = (probs > best_t).astype(int)

print("\nConfusion Matrix (best threshold):")
print(confusion_matrix(y_true, y_pred_best))

print("\nClassification Report (best threshold):")
print(classification_report(y_true, y_pred_best, digits=4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: facebook/deit-small-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 384]) vs model:torch.Size([2, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Silver files: 3655
Non-silver files: 16129
Train size: 5118
Val size: 1462
Test size: 730


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation Results:
eval_loss: 0.9056
eval_accuracy: 0.8694
eval_precision: 0.8782
eval_recall: 0.8577
eval_f1: 0.8678
eval_runtime: 18.4551
eval_samples_per_second: 79.2190
eval_steps_per_second: 4.9850
epoch: 9.0000

Confusion Matrix (threshold = 0.50):
[[323  42]
 [ 50 315]]

Classification Report (threshold = 0.50):
              precision    recall  f1-score   support

           0     0.8660    0.8849    0.8753       365
           1     0.8824    0.8630    0.8726       365

    accuracy                         0.8740       730
   macro avg     0.8742    0.8740    0.8740       730
weighted avg     0.8742    0.8740    0.8740       730

t=0.20 | F1=0.87 | P=0.87 | R=0.87
t=0.21 | F1=0.87 | P=0.87 | R=0.87
t=0.22 | F1=0.87 | P=0.87 | R=0.87
t=0.23 | F1=0.87 | P=0.87 | R=0.87
t=0.24 | F1=0.87 | P=0.87 | R=0.87
t=0.25 | F1=0.87 | P=0.87 | R=0.87
t=0.26 | F1=0.87 | P=0.87 | R=0.87
t=0.27 | F1=0.87 | P=0.87 | R=0.87
t=0.28 | F1=0.87 | P=0.88 | R=0.87
t=0.29 | F1=0.87 | P=0.88 | R=0.87


##Model 2: CNN (RESNET-50)
This code file implements a CNN-based image classification pipeline using ResNet-50 to classify images as silver or non-silver.It loads and balances the dataset from Google Drive, splits it into training, validation, and test sets, and uses a custom dataset class for preprocessing and light augmentation. The pretrained ResNet-50 model is fine-tuned using the Hugging Face Trainer with early stopping and evaluated using accuracy, precision, recall, and F1 score.
The model is tested, and performance is analyzed using a confusion matrix and classification report, with additional threshold tuning applied to optimize F1 score.

In [3]:
# =========================
# Install
# =========================
!pip install -q transformers accelerate evaluate pillow scikit-learn timm

# =========================
# Imports
# =========================
import os
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from transformers import (
    AutoImageProcessor,
    ResNetForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# =========================
# Google Drive
# =========================
drive.mount("/content/drive")

# =========================
# Paths
# =========================
silver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/silver_images"
nonsilver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/non_silver_images"

SEED = 42
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================
# Checkpoint
# =========================
CHECKPOINT = "microsoft/resnet-50"

# =========================
# Processor + model
# =========================
image_processor = AutoImageProcessor.from_pretrained(CHECKPOINT, use_fast=True)

model = ResNetForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label={0: "non_silver", 1: "silver"},
    label2id={"non_silver": 0, "silver": 1},
    ignore_mismatched_sizes=True,   # only classifier head is resized
)

# =========================
# File lists
# =========================
silver_files = [
    f for f in os.listdir(silver_dir)
    if os.path.isfile(os.path.join(silver_dir, f))
]

nonsilver_files = [
    f for f in os.listdir(nonsilver_dir)
    if os.path.isfile(os.path.join(nonsilver_dir, f))
]

print("Silver files:", len(silver_files))
print("Non-silver files:", len(nonsilver_files))

# =========================
# Sample equal non-silver
# =========================
sampled_nonsilver_files = random.sample(nonsilver_files, len(silver_files))

silver_paths = [os.path.join(silver_dir, f) for f in silver_files]
nonsilver_paths = [os.path.join(nonsilver_dir, f) for f in sampled_nonsilver_files]

random.shuffle(silver_paths)
random.shuffle(nonsilver_paths)

# =========================
# Split each class separately
# =========================
n_per_class = len(silver_paths)

test_count = int(n_per_class * TEST_SPLIT)
val_count = int(n_per_class * VAL_SPLIT)

silver_test = silver_paths[:test_count]
silver_val = silver_paths[test_count:test_count + val_count]
silver_train = silver_paths[test_count + val_count:]

nonsilver_test = nonsilver_paths[:test_count]
nonsilver_val = nonsilver_paths[test_count:test_count + val_count]
nonsilver_train = nonsilver_paths[test_count + val_count:]

# =========================
# Combine splits
# =========================
train_paths = silver_train + nonsilver_train
train_labels = [1] * len(silver_train) + [0] * len(nonsilver_train)

val_paths = silver_val + nonsilver_val
val_labels = [1] * len(silver_val) + [0] * len(nonsilver_val)

test_paths = silver_test + nonsilver_test
test_labels = [1] * len(silver_test) + [0] * len(nonsilver_test)

# =========================
# Shuffle within each split
# =========================
train_combined = list(zip(train_paths, train_labels))
val_combined = list(zip(val_paths, val_labels))
test_combined = list(zip(test_paths, test_labels))

random.shuffle(train_combined)
random.shuffle(val_combined)
random.shuffle(test_combined)

train_paths, train_labels = zip(*train_combined)
val_paths, val_labels = zip(*val_combined)
test_paths, test_labels = zip(*test_combined)

train_paths, train_labels = list(train_paths), list(train_labels)
val_paths, val_labels = list(val_paths), list(val_labels)
test_paths, test_labels = list(test_paths), list(test_labels)

print("Train size:", len(train_paths))
print("Val size:", len(val_paths))
print("Test size:", len(test_paths))

# =========================
# Dataset
# =========================
class SilverDataset(Dataset):
    def __init__(self, paths, labels, processor, train=False):
        self.paths = paths
        self.labels = labels
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]

        image = Image.open(path).convert("RGB")

        # light augmentation for training
        if self.train:
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)

        encoded = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

train_ds = SilverDataset(train_paths, train_labels, image_processor, train=True)
val_ds = SilverDataset(val_paths, val_labels, image_processor, train=False)
test_ds = SilverDataset(test_paths, test_labels, image_processor, train=False)

# =========================
# Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    p = precision_score(labels, preds, zero_division=0)
    r = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
    }

# =========================
# Training arguments
# =========================
training_args = TrainingArguments(
    output_dir="./resnet_silver_output",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=160,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

# =========================
# Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================
# Train
# =========================
trainer.train()

# =========================
# Validation results
# =========================
val_results = trainer.evaluate()
print("\nValidation Results:")
for k, v in val_results.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# =========================
# Test prediction
# =========================
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

# =========================
# Classification report at threshold 0.5
# =========================
y_pred = (probs > 0.5).astype(int)

print("\nConfusion Matrix (threshold = 0.50):")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report (threshold = 0.50):")
print(classification_report(y_true, y_pred, digits=4))

# =========================
# Threshold tuning
# =========================
best_t, best_f1 = 0.5, 0.0

for t in np.arange(0.20, 0.70, 0.01):
    preds = (probs > t).astype(int)

    f1 = f1_score(y_true, preds)
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)

    print(f"t={t:.2f} | F1={f1:.2f} | P={p:.2f} | R={r:.2f}")

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"\nBest threshold: {best_t:.2f} | Best F1: {best_f1:.2f}")

y_pred_best = (probs > best_t).astype(int)

print("\nConfusion Matrix (best threshold):")
print(confusion_matrix(y_true, y_pred_best))

print("\nClassification Report (best threshold):")
print(classification_report(y_true, y_pred_best, digits=4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation Results:
eval_loss: 0.3664
eval_accuracy: 0.8550
eval_precision: 0.8569
eval_recall: 0.8523
eval_f1: 0.8546
eval_runtime: 16.3314
eval_samples_per_second: 89.5210
eval_steps_per_second: 2.8170
epoch: 10.0000

Confusion Matrix (threshold = 0.50):
[[318  47]
 [ 61 304]]

Classification Report (threshold = 0.50):
              precision    recall  f1-score   support

           0     0.8391    0.8712    0.8548       365
           1     0.8661    0.8329    0.8492       365

    accuracy                         0.8521       730
   macro avg     0.8526    0.8521    0.8520       730
weighted avg     0.8526    0.8521    0.8520       730

t=0.20 | F1=0.83 | P=0.77 | R=0.90
t=0.21 | F1=0.83 | P=0.77 | R=0.89
t=0.22 | F1=0.83 | P=0.77 | R=0.89
t=0.23 | F1=0.82 | P=0.77 | R=0.88
t=0.24 | F1=0.83 | P=0.78 | R=0.88
t=0.25 | F1=0.83 | P=0.79 | R=0.88
t=0.26 | F1=0.83 | P=0.79 | R=0.88
t=0.27 | F1=0.84 | P=0.79 | R=0.88
t=0.28 | F1=0.84 | P=0.80 | R=0.88
t=0.29 | F1=0.84 | P=0.80 | R=0.88

##Model 3: CNN (DenseNet-121)
This code file implements a DenseNet-based image classification pipeline to classify images as silver or non-silver. It loads and balances the dataset, splits it into training, validation, and test sets, and uses a custom dataset class for preprocessing and light augmentation. A pretrained DenseNet-121 model is fine-tuned using the Hugging Face Trainer with early stopping and evaluated using accuracy, precision, recall, and F1 score. The model is tested, and performance is analyzed using a confusion matrix and classification report, with additional threshold tuning applied to optimize F1 score.

In [4]:
# =========================
# Install
# =========================
!pip install -q transformers accelerate evaluate pillow scikit-learn timm

# =========================
# Imports
# =========================
import os
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# =========================
# Google Drive
# =========================
drive.mount("/content/drive")

# =========================
# Paths
# =========================
silver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/silver_images"
nonsilver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/non_silver_images"

SEED = 42
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================
# Checkpoint
# =========================
CHECKPOINT = "timm/densenet121.tv_in1k"

# =========================
# Processor + model
# =========================
image_processor = AutoImageProcessor.from_pretrained(CHECKPOINT, use_fast=True)

model = AutoModelForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label={0: "non_silver", 1: "silver"},
    label2id={"non_silver": 0, "silver": 1},
    ignore_mismatched_sizes=True,   # only classifier head is resized
)

# =========================
# File lists
# =========================
silver_files = [
    f for f in os.listdir(silver_dir)
    if os.path.isfile(os.path.join(silver_dir, f))
]

nonsilver_files = [
    f for f in os.listdir(nonsilver_dir)
    if os.path.isfile(os.path.join(nonsilver_dir, f))
]

print("Silver files:", len(silver_files))
print("Non-silver files:", len(nonsilver_files))

# =========================
# Sample equal non-silver
# =========================
sampled_nonsilver_files = random.sample(nonsilver_files, len(silver_files))

silver_paths = [os.path.join(silver_dir, f) for f in silver_files]
nonsilver_paths = [os.path.join(nonsilver_dir, f) for f in sampled_nonsilver_files]

random.shuffle(silver_paths)
random.shuffle(nonsilver_paths)

# =========================
# Split each class separately
# =========================
n_per_class = len(silver_paths)

test_count = int(n_per_class * TEST_SPLIT)
val_count = int(n_per_class * VAL_SPLIT)

silver_test = silver_paths[:test_count]
silver_val = silver_paths[test_count:test_count + val_count]
silver_train = silver_paths[test_count + val_count:]

nonsilver_test = nonsilver_paths[:test_count]
nonsilver_val = nonsilver_paths[test_count:test_count + val_count]
nonsilver_train = nonsilver_paths[test_count + val_count:]

# =========================
# Combine splits
# =========================
train_paths = silver_train + nonsilver_train
train_labels = [1] * len(silver_train) + [0] * len(nonsilver_train)

val_paths = silver_val + nonsilver_val
val_labels = [1] * len(silver_val) + [0] * len(nonsilver_val)

test_paths = silver_test + nonsilver_test
test_labels = [1] * len(silver_test) + [0] * len(nonsilver_test)

# =========================
# Shuffle within each split
# =========================
train_combined = list(zip(train_paths, train_labels))
val_combined = list(zip(val_paths, val_labels))
test_combined = list(zip(test_paths, test_labels))

random.shuffle(train_combined)
random.shuffle(val_combined)
random.shuffle(test_combined)

train_paths, train_labels = zip(*train_combined)
val_paths, val_labels = zip(*val_combined)
test_paths, test_labels = zip(*test_combined)

train_paths, train_labels = list(train_paths), list(train_labels)
val_paths, val_labels = list(val_paths), list(val_labels)
test_paths, test_labels = list(test_paths), list(test_labels)

print("Train size:", len(train_paths))
print("Val size:", len(val_paths))
print("Test size:", len(test_paths))

# =========================
# Dataset
# =========================
class SilverDataset(Dataset):
    def __init__(self, paths, labels, processor, train=False):
        self.paths = paths
        self.labels = labels
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]

        image = Image.open(path).convert("RGB")

        # light augmentation for training
        if self.train:
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)

        encoded = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

train_ds = SilverDataset(train_paths, train_labels, image_processor, train=True)
val_ds = SilverDataset(val_paths, val_labels, image_processor, train=False)
test_ds = SilverDataset(test_paths, test_labels, image_processor, train=False)

# =========================
# Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    p = precision_score(labels, preds, zero_division=0)
    r = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
    }

# =========================
# Training arguments
# =========================
training_args = TrainingArguments(
    output_dir="./densenet_silver_output",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=160,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

# =========================
# Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================
# Train
# =========================
trainer.train()

# =========================
# Validation results
# =========================
val_results = trainer.evaluate()
print("\nValidation Results:")
for k, v in val_results.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# =========================
# Test prediction
# =========================
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

# =========================
# Classification report at threshold 0.5
# =========================
y_pred = (probs > 0.5).astype(int)

print("\nConfusion Matrix (threshold = 0.50):")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report (threshold = 0.50):")
print(classification_report(y_true, y_pred, digits=4))

# =========================
# Threshold tuning
# =========================
best_t, best_f1 = 0.5, 0.0

for t in np.arange(0.20, 0.70, 0.01):
    preds = (probs > t).astype(int)

    f1 = f1_score(y_true, preds)
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)

    print(f"t={t:.2f} | F1={f1:.2f} | P={p:.2f} | R={r:.2f}")

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"\nBest threshold: {best_t:.2f} | Best F1: {best_f1:.2f}")

y_pred_best = (probs > best_t).astype(int)

print("\nConfusion Matrix (best threshold):")
print(confusion_matrix(y_true, y_pred_best))

print("\nClassification Report (best threshold):")
print(classification_report(y_true, y_pred_best, digits=4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

TimmWrapperForImageClassification LOAD REPORT from: timm/densenet121.tv_in1k
Key                          | Status   |                                                                                          
-----------------------------+----------+------------------------------------------------------------------------------------------
timm_model.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])            
timm_model.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 1024]) vs model:torch.Size([2, 1024])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation Results:
eval_loss: 0.4403
eval_accuracy: 0.8632
eval_precision: 0.8489
eval_recall: 0.8837
eval_f1: 0.8660
eval_runtime: 19.2711
eval_samples_per_second: 75.8650
eval_steps_per_second: 2.3870
epoch: 8.0000

Confusion Matrix (threshold = 0.50):
[[314  51]
 [ 47 318]]

Classification Report (threshold = 0.50):
              precision    recall  f1-score   support

           0     0.8698    0.8603    0.8650       365
           1     0.8618    0.8712    0.8665       365

    accuracy                         0.8658       730
   macro avg     0.8658    0.8658    0.8657       730
weighted avg     0.8658    0.8658    0.8657       730

t=0.20 | F1=0.85 | P=0.81 | R=0.90
t=0.21 | F1=0.86 | P=0.82 | R=0.90
t=0.22 | F1=0.86 | P=0.82 | R=0.90
t=0.23 | F1=0.86 | P=0.82 | R=0.90
t=0.24 | F1=0.85 | P=0.82 | R=0.89
t=0.25 | F1=0.86 | P=0.82 | R=0.89
t=0.26 | F1=0.86 | P=0.82 | R=0.89
t=0.27 | F1=0.86 | P=0.83 | R=0.89
t=0.28 | F1=0.86 | P=0.83 | R=0.89
t=0.29 | F1=0.86 | P=0.83 | R=0.89


##Model 4: CNN (EfficientNet-B0)
This code file implements a binary image classification pipeline using a convolutional neural network (EfficientNet-B0) to classify images as silver or non-silver. It loads images from Google Drive, balances the dataset by sampling equal classes, and splits the data into training, validation, and test sets. A custom dataset class handles image loading, preprocessing, and light augmentation. The pretrained EfficientNet-B0 model is fine-tuned using Hugging Face’s Trainer with early stopping and evaluated using accuracy, precision, recall, and F1 score. The model is tested, and performance is analyzed using a confusion matrix and classification report, with additional threshold tuning applied to optimize F1 score.

In [5]:
# =========================
# Install
# =========================
!pip install -q transformers accelerate evaluate pillow scikit-learn timm

# =========================
# Imports
# =========================
import os
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# =========================
# Google Drive
# =========================
drive.mount("/content/drive")

# =========================
# Paths
# =========================
silver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/silver_images"
nonsilver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/non_silver_images"

SEED = 42
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================
# Checkpoint
# =========================
CHECKPOINT = "google/efficientnet-b0"

# =========================
# Processor + model
# =========================
image_processor = AutoImageProcessor.from_pretrained(CHECKPOINT, use_fast=True)

model = AutoModelForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label={0: "non_silver", 1: "silver"},
    label2id={"non_silver": 0, "silver": 1},
    ignore_mismatched_sizes=True,   # only classifier head is resized
)

# =========================
# File lists
# =========================
silver_files = [
    f for f in os.listdir(silver_dir)
    if os.path.isfile(os.path.join(silver_dir, f))
]

nonsilver_files = [
    f for f in os.listdir(nonsilver_dir)
    if os.path.isfile(os.path.join(nonsilver_dir, f))
]

print("Silver files:", len(silver_files))
print("Non-silver files:", len(nonsilver_files))

# =========================
# Sample equal non-silver
# =========================
sampled_nonsilver_files = random.sample(nonsilver_files, len(silver_files))

silver_paths = [os.path.join(silver_dir, f) for f in silver_files]
nonsilver_paths = [os.path.join(nonsilver_dir, f) for f in sampled_nonsilver_files]

random.shuffle(silver_paths)
random.shuffle(nonsilver_paths)

# =========================
# Split each class separately
# =========================
n_per_class = len(silver_paths)

test_count = int(n_per_class * TEST_SPLIT)
val_count = int(n_per_class * VAL_SPLIT)

silver_test = silver_paths[:test_count]
silver_val = silver_paths[test_count:test_count + val_count]
silver_train = silver_paths[test_count + val_count:]

nonsilver_test = nonsilver_paths[:test_count]
nonsilver_val = nonsilver_paths[test_count:test_count + val_count]
nonsilver_train = nonsilver_paths[test_count + val_count:]

# =========================
# Combine splits
# =========================
train_paths = silver_train + nonsilver_train
train_labels = [1] * len(silver_train) + [0] * len(nonsilver_train)

val_paths = silver_val + nonsilver_val
val_labels = [1] * len(silver_val) + [0] * len(nonsilver_val)

test_paths = silver_test + nonsilver_test
test_labels = [1] * len(silver_test) + [0] * len(nonsilver_test)

# =========================
# Shuffle within each split
# =========================
train_combined = list(zip(train_paths, train_labels))
val_combined = list(zip(val_paths, val_labels))
test_combined = list(zip(test_paths, test_labels))

random.shuffle(train_combined)
random.shuffle(val_combined)
random.shuffle(test_combined)

train_paths, train_labels = zip(*train_combined)
val_paths, val_labels = zip(*val_combined)
test_paths, test_labels = zip(*test_combined)

train_paths, train_labels = list(train_paths), list(train_labels)
val_paths, val_labels = list(val_paths), list(val_labels)
test_paths, test_labels = list(test_paths), list(test_labels)

print("Train size:", len(train_paths))
print("Val size:", len(val_paths))
print("Test size:", len(test_paths))

# =========================
# Dataset
# =========================
class SilverDataset(Dataset):
    def __init__(self, paths, labels, processor, train=False):
        self.paths = paths
        self.labels = labels
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]

        image = Image.open(path).convert("RGB")

        # light augmentation for training
        if self.train:
            if random.random() > 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)

        encoded = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

train_ds = SilverDataset(train_paths, train_labels, image_processor, train=True)
val_ds = SilverDataset(val_paths, val_labels, image_processor, train=False)
test_ds = SilverDataset(test_paths, test_labels, image_processor, train=False)

# =========================
# Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    p = precision_score(labels, preds, zero_division=0)
    r = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
    }

# =========================
# Training arguments
# =========================
training_args = TrainingArguments(
    output_dir="./efficientnet_silver_output",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=160,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

# =========================
# Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================
# Train
# =========================
trainer.train()

# =========================
# Validation results
# =========================
val_results = trainer.evaluate()
print("\nValidation Results:")
for k, v in val_results.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

# =========================
# Test prediction
# =========================
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

# =========================
# Classification report at threshold 0.5
# =========================
y_pred = (probs > 0.5).astype(int)

print("\nConfusion Matrix (threshold = 0.50):")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report (threshold = 0.50):")
print(classification_report(y_true, y_pred, digits=4))

# =========================
# Threshold tuning
# =========================
best_t, best_f1 = 0.5, 0.0

for t in np.arange(0.20, 0.70, 0.01):
    preds = (probs > t).astype(int)

    f1 = f1_score(y_true, preds)
    p = precision_score(y_true, preds, zero_division=0)
    r = recall_score(y_true, preds, zero_division=0)

    print(f"t={t:.2f} | F1={f1:.2f} | P={p:.2f} | R={r:.2f}")

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print(f"\nBest threshold: {best_t:.2f} | Best F1: {best_f1:.2f}")

y_pred_best = (probs > best_t).astype(int)

print("\nConfusion Matrix (best threshold):")
print(confusion_matrix(y_true, y_pred_best))

print("\nClassification Report (best threshold):")
print(classification_report(y_true, y_pred_best, digits=4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


preprocessor_config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation Results:
eval_loss: 0.4799
eval_accuracy: 0.8523
eval_precision: 0.8393
eval_recall: 0.8714
eval_f1: 0.8550
eval_runtime: 12.6234
eval_samples_per_second: 115.8160
eval_steps_per_second: 3.6440
epoch: 9.0000

Confusion Matrix (threshold = 0.50):
[[312  53]
 [ 52 313]]

Classification Report (threshold = 0.50):
              precision    recall  f1-score   support

           0     0.8571    0.8548    0.8560       365
           1     0.8552    0.8575    0.8564       365

    accuracy                         0.8562       730
   macro avg     0.8562    0.8562    0.8562       730
weighted avg     0.8562    0.8562    0.8562       730

t=0.20 | F1=0.86 | P=0.81 | R=0.92
t=0.21 | F1=0.86 | P=0.81 | R=0.92
t=0.22 | F1=0.86 | P=0.82 | R=0.92
t=0.23 | F1=0.86 | P=0.82 | R=0.91
t=0.24 | F1=0.86 | P=0.82 | R=0.91
t=0.25 | F1=0.86 | P=0.82 | R=0.90
t=0.26 | F1=0.86 | P=0.82 | R=0.90
t=0.27 | F1=0.86 | P=0.82 | R=0.90
t=0.28 | F1=0.86 | P=0.82 | R=0.90
t=0.29 | F1=0.86 | P=0.83 | R=0.90

##Model 6: Transformer (ViT Baseline)

This code file implements a binary image classification pipeline using a standard transformer model (Vision Transformer, ViT) to classify images as silver or non-silver. It loads images from Google Drive, balances the dataset by sampling equal classes, and splits the data into training, validation, and test sets. A custom dataset class handles image loading, preprocessing, and light augmentation. The model is fine-tuned using Hugging Face’s Trainer with early stopping and evaluated using accuracy, precision, recall, and F1 score. The model is tested, and performance is analyzed using a confusion matrix and classification report, with additional threshold tuning applied to optimize F1 score.

In [7]:
# =========================
# Install
# =========================
!pip install -q transformers accelerate evaluate pillow scikit-learn

# =========================
# Imports
# =========================
import os
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

from transformers import (
    AutoImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

# =========================
# Google Drive
# =========================
drive.mount("/content/drive")

# =========================
# Paths
# =========================
silver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/silver_images"
nonsilver_dir = "/content/drive/MyDrive/silver_detection/cleaned_dataset/non_silver_images"

SEED = 42
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# =========================
# ✅ BASELINE TRANSFORMER
# =========================
CHECKPOINT = "google/vit-base-patch16-224"

# =========================
# Processor + model
# =========================
image_processor = AutoImageProcessor.from_pretrained(CHECKPOINT)

model = ViTForImageClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label={0: "non_silver", 1: "silver"},
    label2id={"non_silver": 0, "silver": 1},
    ignore_mismatched_sizes=True,
)

# =========================
# File lists
# =========================
silver_files = [
    f for f in os.listdir(silver_dir)
    if os.path.isfile(os.path.join(silver_dir, f))
]

nonsilver_files = [
    f for f in os.listdir(nonsilver_dir)
    if os.path.isfile(os.path.join(nonsilver_dir, f))
]

print("Silver files:", len(silver_files))
print("Non-silver files:", len(nonsilver_files))

# =========================
# Balance dataset
# =========================
sampled_nonsilver_files = random.sample(nonsilver_files, len(silver_files))

silver_paths = [os.path.join(silver_dir, f) for f in silver_files]
nonsilver_paths = [os.path.join(nonsilver_dir, f) for f in sampled_nonsilver_files]

random.shuffle(silver_paths)
random.shuffle(nonsilver_paths)

# =========================
# Split dataset
# =========================
n_per_class = len(silver_paths)

test_count = int(n_per_class * TEST_SPLIT)
val_count = int(n_per_class * VAL_SPLIT)

silver_test = silver_paths[:test_count]
silver_val = silver_paths[test_count:test_count + val_count]
silver_train = silver_paths[test_count + val_count:]

nonsilver_test = nonsilver_paths[:test_count]
nonsilver_val = nonsilver_paths[test_count:test_count + val_count]
nonsilver_train = nonsilver_paths[test_count + val_count:]

# Combine
train_paths = silver_train + nonsilver_train
train_labels = [1]*len(silver_train) + [0]*len(nonsilver_train)

val_paths = silver_val + nonsilver_val
val_labels = [1]*len(silver_val) + [0]*len(nonsilver_val)

test_paths = silver_test + nonsilver_test
test_labels = [1]*len(silver_test) + [0]*len(nonsilver_test)

# Shuffle
def shuffle_data(paths, labels):
    combined = list(zip(paths, labels))
    random.shuffle(combined)
    return zip(*combined)

train_paths, train_labels = shuffle_data(train_paths, train_labels)
val_paths, val_labels = shuffle_data(val_paths, val_labels)
test_paths, test_labels = shuffle_data(test_paths, test_labels)

train_paths, train_labels = list(train_paths), list(train_labels)
val_paths, val_labels = list(val_paths), list(val_labels)
test_paths, test_labels = list(test_paths), list(test_labels)

print("Train:", len(train_paths), "Val:", len(val_paths), "Test:", len(test_paths))

# =========================
# Dataset
# =========================
class SilverDataset(Dataset):
    def __init__(self, paths, labels, processor, train=False):
        self.paths = paths
        self.labels = labels
        self.processor = processor
        self.train = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        label = self.labels[idx]

        if self.train and random.random() > 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)

        encoded = self.processor(images=image, return_tensors="pt")

        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

train_ds = SilverDataset(train_paths, train_labels, image_processor, train=True)
val_ds = SilverDataset(val_paths, val_labels, image_processor, train=False)
test_ds = SilverDataset(test_paths, test_labels, image_processor, train=False)

# =========================
# Metrics
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }

# =========================
# Training arguments
# =========================
training_args = TrainingArguments(
    output_dir="./vit_baseline_output",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

# =========================
# Trainer
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# =========================
# Train
# =========================
trainer.train()

# =========================
# Evaluate
# =========================
val_results = trainer.evaluate()
print("\nValidation Results:", val_results)

pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_true = pred_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

# =========================
# Threshold tuning
# =========================
best_t, best_f1 = 0.5, 0

for t in np.arange(0.2, 0.7, 0.01):
    preds = (probs > t).astype(int)
    f1 = f1_score(y_true, preds)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"Best threshold: {best_t:.2f} | F1: {best_f1:.2f}")

y_pred = (probs > best_t).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Silver files: 3655
Non-silver files: 16129
Train: 5118 Val: 1462 Test: 730


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation Results: {'eval_loss': 0.6994368433952332, 'eval_accuracy': 0.8809849521203831, 'eval_precision': 0.8928067700987306, 'eval_recall': 0.86593707250342, 'eval_f1': 0.8791666666666667, 'eval_runtime': 23.3293, 'eval_samples_per_second': 62.668, 'eval_steps_per_second': 3.944, 'epoch': 9.0}
Best threshold: 0.68 | F1: 0.89

Confusion Matrix:
[[337  28]
 [ 53 312]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8641    0.9233    0.8927       365
           1     0.9176    0.8548    0.8851       365

    accuracy                         0.8890       730
   macro avg     0.8909    0.8890    0.8889       730
weighted avg     0.8909    0.8890    0.8889       730

